In [129]:
import pandas as pd
import numpy as np
import quandl
import math, datetime
from sklearn import preprocessing, cross_validation, svm
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
from matplotlib import style

style.use('ggplot')

In [67]:
df = quandl.get('WIKI/GOOG')
len(df)

861

In [68]:
df.head()

,Open,High,Low,Close,Volume,Ex-Dividend,Split Ratio,Adj. Open,Adj. High,Adj. Low,Adj. Close,Adj. Volume
Date,,,,,,,,,,,,
2014-03-27,568.000,568.00,552.92,558.46,13100.0,0.0,1.0,568.000,568.00,552.92,558.46,13100.0
2014-03-28,561.200,566.43,558.67,559.99,41100.0,0.0,1.0,561.200,566.43,558.67,559.99,41100.0
2014-03-31,566.890,567.00,556.93,556.97,10800.0,0.0,1.0,566.890,567.00,556.93,556.97,10800.0
2014-04-01,558.710,568.45,558.71,567.16,7900.0,0.0,1.0,558.710,568.45,558.71,567.16,7900.0
2014-04-02,565.106,604.83,562.19,567.00,146700.0,0.0,1.0,565.106,604.83,562.19,567.00,146700.0


In [69]:
df=df[['Adj. Open', 'Adj. High', 'Adj. Low','Adj. Close', 'Adj. Volume']]

In [70]:
print df.head()

            Adj. Open  Adj. High  Adj. Low  Adj. Close  Adj. Volume
Date                                                               
2014-03-27    568.000     568.00    552.92      558.46      13100.0
2014-03-28    561.200     566.43    558.67      559.99      41100.0
2014-03-31    566.890     567.00    556.93      556.97      10800.0
2014-04-01    558.710     568.45    558.71      567.16       7900.0
2014-04-02    565.106     604.83    562.19      567.00     146700.0


In [71]:

df['PCT_Change'] = (df['Adj. Close'] - df['Adj. Open'])*100.0/df['Adj. Open']
df['HL_PCT'] = (df['Adj. High'] - df['Adj. Close'])*100.0/df['Adj. Close']


In [72]:
df = df[['Adj. Close', 'HL_PCT','PCT_Change','Adj. Volume']]
df.head()

,Adj. Close,HL_PCT,PCT_Change,Adj. Volume
Date,,,,
2014-03-27,558.46,1.708269,-1.679577,13100.0
2014-03-28,559.99,1.150021,-0.215609,41100.0
2014-03-31,556.97,1.800815,-1.749899,10800.0
2014-04-01,567.16,0.227449,1.512413,7900.0
2014-04-02,567.00,6.671958,0.335158,146700.0


In [73]:
forecast_col = 'Adj. Close'

In [74]:
df.fillna(-9999,inplace=True)

In [75]:
forecast_out=int(math.ceil(0.01*len(df)))


In [76]:
print forecast_out

9


In [77]:
df['label'] =df[forecast_col].shift(-forecast_out)
print df[850:]


            Adj. Close    HL_PCT  PCT_Change  Adj. Volume   label
Date                                                             
2017-08-10      907.24  1.324897   -1.123644    1722296.0  927.00
2017-08-11      914.39  0.370739    0.707072    1190458.0  921.28
2017-08-14      922.67  0.216545    0.015176    1047828.0     NaN
2017-08-15      922.22  0.469508   -0.217478     873070.0     NaN
2017-08-16      926.96  0.619228    0.180484     988604.0     NaN
2017-08-17      910.98  1.743178   -1.598652    1218963.0     NaN
2017-08-18      910.67  0.505672    0.039547    1333572.0     NaN
2017-08-21      906.66  0.699270   -0.367033     932903.0     NaN
2017-08-22      924.69  0.126529    1.311465    1145571.0     NaN
2017-08-23      927.00  0.316073    0.549933    1077809.0     NaN
2017-08-24      921.28  1.037687   -0.794693    1218875.0     NaN


In [78]:
import sklearn

In [79]:
df.dropna(inplace=True)
print df.tail()
df.columns

            Adj. Close    HL_PCT  PCT_Change  Adj. Volume   label
Date                                                             
2017-08-07      929.36  0.251786    0.032291    1012296.0  910.67
2017-08-08      926.79  0.973683   -0.032359    1039394.0  906.66
2017-08-09      922.90  0.333731    0.248748    1169431.0  924.69
2017-08-10      907.24  1.324897   -1.123644    1722296.0  927.00
2017-08-11      914.39  0.370739    0.707072    1190458.0  921.28


Index([u'Adj. Close', u'HL_PCT', u'PCT_Change', u'Adj. Volume', u'label'], dtype='object')

In [80]:
x = np.array(df.drop(['label'],1))
y = np.array(df['label'])

In [81]:
len(x), len(y)
print x

[[  5.58460000e+02   1.70826917e+00  -1.67957746e+00   1.31000000e+04]
 [  5.59990000e+02   1.15002054e+00  -2.15609408e-01   4.11000000e+04]
 [  5.56970000e+02   1.80081512e+00  -1.74989857e+00   1.08000000e+04]
 ..., 
 [  9.22900000e+02   3.33730632e-01   2.48748113e-01   1.16943100e+06]
 [  9.07240000e+02   1.32489749e+00  -1.12364449e+00   1.72229600e+06]
 [  9.14390000e+02   3.70738963e-01   7.07071820e-01   1.19045800e+06]]


In [92]:
X = preprocessing.scale(x)

In [84]:
print X


[[  5.58460000e+02   1.70826917e+00  -1.67957746e+00   1.31000000e+04]
 [  5.59990000e+02   1.15002054e+00  -2.15609408e-01   4.11000000e+04]
 [  5.56970000e+02   1.80081512e+00  -1.74989857e+00   1.08000000e+04]
 ..., 
 [  9.41530000e+02   2.44283241e-01   1.30514310e+00   1.77507600e+06]
 [  9.30500000e+02   1.40677055e+00  -1.20927072e+00   1.95271600e+06]
 [  9.30830000e+02   7.10870943e-01  -1.66241232e-01   1.21155300e+06]]


In [85]:
len(X)

844

In [86]:
df.dropna(inplace=True)

In [87]:
len(df)

852

In [88]:
y=np.array(df['label'])

In [89]:
len(y)

852

In [93]:
x_train, x_test, y_train, y_test = cross_validation.train_test_split(X,y,test_size=0.2)

In [95]:
len(x_test), len(y_test), len(x_train), len(y_train)

(171, 171, 681, 681)

In [125]:
clf = LinearRegression(n_jobs=-1)
clf.fit(x_train,y_train)

LinearRegression(copy_X=True, fit_intercept=True, n_jobs=-1, normalize=False)

In [126]:
accuracy = clf.score(x_test,y_test)

In [127]:
print accuracy

0.958775332077


In [133]:
X = np.array(df.drop(['label'],1))
X = preprocessing.scale(X)
X = X[:-forecast_out]
X_lately = x[-forecast_out:]


In [134]:
len(X)

843

In [135]:
len(X_lately)

9

In [136]:
accuracy = clf.score(x_test, y_test)

In [137]:
print accuracy

0.958775332077


In [141]:
forecast_set = clf.predict(X_lately)

In [142]:
print(forecast_set, forecast_out)

(array([-2299549.65039925, -3507665.05544162, -2235701.23354021,
       -1915615.96328878, -1901727.04650593, -1956184.45034348,
       -2216432.97379551, -3322786.09589254, -2259529.32220767]), 9)


In [145]:
df['Forecast'] = np.nan
last_date = df.iloc[-1].name

In [147]:
print df['Forecast']
print last_date

Date
2014-03-27   NaN
2014-03-28   NaN
2014-03-31   NaN
2014-04-01   NaN
2014-04-02   NaN
2014-04-03   NaN
2014-04-04   NaN
2014-04-07   NaN
2014-04-08   NaN
2014-04-09   NaN
2014-04-10   NaN
2014-04-11   NaN
2014-04-14   NaN
2014-04-15   NaN
2014-04-16   NaN
2014-04-17   NaN
2014-04-21   NaN
2014-04-22   NaN
2014-04-23   NaN
2014-04-24   NaN
2014-04-25   NaN
2014-04-28   NaN
2014-04-29   NaN
2014-04-30   NaN
2014-05-01   NaN
2014-05-02   NaN
2014-05-05   NaN
2014-05-06   NaN
2014-05-07   NaN
2014-05-08   NaN
              ..
2017-06-30   NaN
2017-07-03   NaN
2017-07-05   NaN
2017-07-06   NaN
2017-07-07   NaN
2017-07-10   NaN
2017-07-11   NaN
2017-07-12   NaN
2017-07-13   NaN
2017-07-14   NaN
2017-07-17   NaN
2017-07-18   NaN
2017-07-19   NaN
2017-07-20   NaN
2017-07-21   NaN
2017-07-24   NaN
2017-07-25   NaN
2017-07-26   NaN
2017-07-27   NaN
2017-07-28   NaN
2017-07-31   NaN
2017-08-01   NaN
2017-08-02   NaN
2017-08-03   NaN
2017-08-04   NaN
2017-08-07   NaN
2017-08-08   NaN
2017-08-0